# 05 · Inference with your exact policy
**15 minutes. Objective:** verify Notebook04's actual checkpoint/export, physically admit that exact actor, then inspect its live onboard inference episode without sending an agent command.

Inference uses fixed weights; admission asks whether measured behavior meets unchanged motion/rest criteria. **The implemented API requires admission before `serve`**, so we admit first, then launch camera serving. This is not a policy substitution. A failed admission remains a useful recorded result, but cannot authorize Notebook06 movement.

No ROS gateway or model starts in this notebook. Zero requested velocity still uses active neural balance; it is not torque-off. Native admission is **not** ROS/perception/CASCADE acceptance.

In [ ]:
from pathlib import Path
import os, sys, json
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Select the host Python 3.12 kernel created by scripts/notebook_setup.sh")
# This kernel is deliberately not a ROS / Kit interpreter.
start = Path(os.environ.get("WORKSHOP_ROOT", Path.cwd())).resolve()
ROOT = next((p for p in (start, *start.parents)
             if (p / "notebooks/workshop.py").is_file()
             and (p / "src/microduck_workshop").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository, or set WORKSHOP_ROOT")
sys.path.insert(0, str(ROOT / "notebooks"))
from policy_workshop import PolicySession, training_bundle, admitted_bundle, read_json, sha256
session = PolicySession(ROOT, work=os.environ.get("WORKSHOP_NOTEBOOK_WORK"))
VALIDATE = os.environ.get("WORKSHOP_NOTEBOOK_MODE") == "validate"
print("Host Python:", sys.version.split()[0])
print("Mode:", "safe validation (hardware SKIPPED)" if VALIDATE else "interactive")

## Predict · 3 minutes
What must happen if `actor.onnx`, `checkpoint.pt`, a raw admission trace, or the control source changes? Why is `passed:true` in a JSON file insufficient?

The native loader recomputes physical verdicts from all retained raw traces. It binds the current source, mode, fixed step, BAM/runtime identity, manifest and actor. Public receipt snapshots without weights/raw traces are not deployment bundles.

In [ ]:
RUN_ADMISSION = False
RUN_INFERENCE = False
ACCEPT_NVIDIA_EULA = False
GPU = os.environ.get("WORKSHOP_GPU", "0")
SIM_IMAGE = os.environ.get("WORKSHOP_SIM_IMAGE", "roscon/microduck-sim:local")
ASSETS = Path(os.environ.get("WORKSHOP_ASSET_ROOT", ROOT / ".cache/assets")).resolve()
SIM_URL = "http://127.0.0.1:8766"

def inspect_training_handoff():
    bundle = session.selection("04")
    session.capabilities(SIM_IMAGE)
    session.verify_bundle(SIM_IMAGE, bundle)
    print("Recomputed Notebook04 identity:", json.dumps(bundle["identity"], indent=2))
session.step("recheck exact checkpoint/export/source", RUN_ADMISSION or RUN_INFERENCE, inspect_training_handoff)

## Task · 7 minutes
Before executing, predict a legitimate reason for admission failure even when export parity is good. Review the unchanged evaluator source; do not alter thresholds.

Opt in to the exact-policy admission job. It performs independent native step/resume and zero/rest controls. It needs a **new** output directory and retains required raw traces. There is no listener or agent authority during this job. The seven-minute deadline includes runtime startup; slower hardware is BLOCKED/incomplete, not a reason to loosen gates.

In [ ]:
def admit_exact_actor():
    bundle = session.selection("04")
    if set(session.state().get("containers", {})) & {"sim", "ros", "train", "admission", "agent"}:
        raise RuntimeError("Clean up your earlier runtime before admission")
    session.gpu_preflight(GPU, accept_eula=ACCEPT_NVIDIA_EULA)
    output = session.new_output("admission")
    state = session.state()
    for stage in list(state["stages"]):
        if stage >= "05":
            del state["stages"][stage]
    state["admission_candidate"] = str(output / "admission")
    session.save(state)
    session.job("admission", SIM_IMAGE,
        ["-m", "microduck_workshop.sim.admit_policy", "--asset-root", "/assets",
         "--policy-manifest", "/policy-run/policy_manifest.json", "--output", "/outputs/admission"],
        timeout=420, memory="32g", cpus=8,
        options=[*session.sim_options(GPU, ASSETS, output), *session.bundle_options(bundle)])
session.step("native exact-policy physical admission", RUN_ADMISSION, admit_exact_actor)

## Observe · admission result
Read the real outcome even if the preceding job failed. Missing receipt is incomplete evidence. A false result blocks serving, without selecting the old VelStand actor. Do not edit the receipt.

**Expected behavior, not invented output:** only a complete unchanged-source bundle can pass canonical validation. The admission hash is added to the same checkpoint/export/manifest identity from04.

In [ ]:
def inspect_admission():
    directory = Path(session.state()["admission_candidate"])
    path = directory / "admission.json"
    if not path.is_file():
        raise RuntimeError("No final admission receipt; inspect the preserved job log")
    receipt = read_json(path)
    print(json.dumps({key: receipt.get(key) for key in ("scope", "passed", "policy_sha256", "cases", "error")}, indent=2))
    trained = session.selection("04")
    bundle = admitted_bundle(ROOT, trained["run"], directory)
    session.verify_bundle(SIM_IMAGE, bundle)  # recompute, do not trust serialized booleans
    state = session.state()
    state["admitted_candidate"] = bundle
    session.save(state)
    print("Admitted bundle identity:", json.dumps(bundle["identity"], indent=2))
session.step("inspect actual admission and raw-trace binding", RUN_ADMISSION or RUN_INFERENCE, inspect_admission)

## Observe · 3 minutes: live inference
After admission, opt in separately to serving. This starts a **fresh** episode, not the introductory03 robot. Both full run and admission directories are mounted read-only. Native `play` refuses an altered/missing policy and has no fallback.

Check live policy/manifest/admission identity before viewing the camera. The checkpoint hash remains checked on disk; the live runtime reports the exported actor, not an alleged loaded optimizer. Identity inspection belongs to the supervisor, **not a model tool**. The camera below is the real onboard image; no overview or world pose is passed to CASCADE.

In [ ]:
def start_inference():
    trained = session.selection("04")
    bundle = admitted_bundle(ROOT, trained["run"], session.state()["admission_candidate"])
    session.gpu_preflight(GPU, accept_eula=ACCEPT_NVIDIA_EULA)
    entry = session.start_policy(SIM_IMAGE, bundle, gpu=GPU, assets=ASSETS, url=SIM_URL,
                                 previous_episode=session.require("03")["episode_id"])
    state = session.state()
    state["inference_episode"] = entry
    session.save(state)
    print("Live identity:", session.live_identity(SIM_URL, bundle, entry["episode_id"]))
session.step("fresh exact-policy native inference episode", RUN_INFERENCE, start_inference)

In [ ]:
def observe_and_record_inference():
    from IPython.display import Image, display
    entry = session.state()["inference_episode"]
    trained = session.selection("04")
    bundle = admitted_bundle(ROOT, trained["run"], entry["admission"])
    if bundle["identity"] != entry["identity"]:
        raise RuntimeError("Inference handoff changed")
    session.live_identity(SIM_URL, bundle, entry["episode_id"])
    path = session.camera_snapshot(SIM_URL, expected_episode=entry["episode_id"], label="05-head")
    display(Image(filename=str(path)))
    session.complete("05", entry, artifacts=[*bundle["files"], path, path.with_name("capture.json")])
session.step("actual onboard image and exact-policy inference checkpoint", RUN_INFERENCE, observe_and_record_inference)

## Debug and Cleanup · 2 minutes
- Failed admission: no serving or agent motion. Continue06's CPU adapter work; discuss the real failure. An explicit offline `play --mode validate --policy-manifest ...` diagnostic without a step profile is a different experiment, not a back door to `serve` or proof of fixed-step admission.
- Hash/source mismatch: refuse. Preserve the failed handoff; reproduce on the approved release.
- Camera/identity mismatch or occupied8766: do not attach to another process, fabricate pixels, or rearm an old episode.
- Timeout: owned cleanup; never automatic retry. Keep logs, weights and traces.

Explain the difference between “same model bytes,” “native physical admission” and “full agent acceptance.” Stop this inference episode before06; the next notebook deliberately creates a new episode after the model is ready.

In [ ]:
def cleanup_inference():
    session.stop("admission")
    session.stop("sim")
session.step("owned inference cleanup before model/gateway", RUN_ADMISSION or RUN_INFERENCE, cleanup_inference)